# CareSync: Bronze to Silver Load

## What does this notebook do?
It takes **one batch of raw data** from the **bronze** layer and loads it into the **silver** layer of the CareSync lakehouse.

```
adb_db_caresync.bronze.<table_name>   ──►   adb_db_caresync.silver.<table_name>
      (raw landed data)                         (clean, query-ready data)
```

It is **generic and reusable**: the same notebook works for any table (hospitals, patients, etc.). You just pass different parameters.

## Parameters (widgets)
| Parameter | What it means | Example |
| --- | --- | --- |
| `table_name` | Name of the table, the same in bronze and silver | `hospitals` |
| `landing_timestamp` | Which batch to pick from bronze (only rows with this exact value are loaded) | `2026-11-08 07:00:00` |
| `load_type` | How to write into silver: `FULL`, `APPEND` or `MERGE` | `MERGE` |
| `merge_keys` | Comma-separated key column(s). **Only needed for `MERGE`** | `hospital_id` |

## The 3 load types, in plain words
| Load type | What happens to the silver table | When to use it |
| --- | --- | --- |
| **FULL** | Everything in silver is **deleted and replaced** with this batch | Small reference tables that are sent in full each time |
| **APPEND** | The batch is **added on top** of what is already there (no checks for duplicates) | Event/log style data where rows never change |
| **MERGE** | **Upsert**: rows whose key already exists are **updated**, new keys are **inserted** | Master data that changes over time (e.g. patient details) |

## Steps in this notebook
1. **Read parameters** from the widgets / job.
2. **Read the batch** from the bronze table using `landing_timestamp`.
3. **Add audit columns** `insert_timestamp` and `update_timestamp`.
4. **Write to silver** using the chosen `load_type`.
5. **Quick check** of a silver table (optional).
6. **Return the record count** to the calling job with `dbutils.notebook.exit`.

Each step prints a short, readable summary so you can follow what happened in the job output.

## Step 1: Read the job parameters
The widgets below receive values either from a **job / parent notebook** (via `dbutils.notebook.run` or job task parameters) or typed in manually when running interactively.

If any required value is empty, the cell prints a warning so you can spot the problem early.

In [0]:
dbutils.widgets.text('landing_timestamp', '')
dbutils.widgets.text('table_name', '')
dbutils.widgets.text('merge_keys', '')
dbutils.widgets.text('load_type', '')

landing_timestamp = dbutils.widgets.get('landing_timestamp')
table_name = dbutils.widgets.get('table_name')
merge_keys = dbutils.widgets.get('merge_keys')
load_type = dbutils.widgets.get('load_type')

print("=" * 70)
print("STEP 1: Job parameters received")
print("=" * 70)
print(f"  Table name        : {table_name or '(empty!)'}")
print(f"  Landing timestamp : {landing_timestamp or '(empty!)'}")
print(f"  Load type         : {load_type or '(empty!)'}   -> expected FULL / APPEND / MERGE")
print(f"  Merge keys        : {merge_keys or '(none - only needed for MERGE)'}")
if not table_name or not landing_timestamp or not load_type:
    print("\n  WARNING: One or more required parameters are empty. "
          "Set them in the widgets above or pass them from the job.")


## Step 2: Read the batch from bronze
Bronze can hold many loads of the same table. Each load is tagged with a `landing_timestamp`.

Here only the rows of **this** load are picked (`landing_timestamp = <parameter>`), so older batches are not loaded again. The cell prints the row count and column list, then shows a preview.

In [0]:
bronze_df = spark.read.table(f'adb_db_caresync.bronze.{table_name}').filter(f"landing_timestamp = '{landing_timestamp}'")

bronze_count = bronze_df.count()

print("=" * 70)
print("STEP 2: Read data from BRONZE")
print("=" * 70)
print(f"  Source table      : adb_db_caresync.bronze.{table_name}")
print(f"  Filter applied    : landing_timestamp = '{landing_timestamp}'")
print(f"  Rows found        : {bronze_count}")
print(f"  Columns ({len(bronze_df.columns)})      : {', '.join(bronze_df.columns)}")
if bronze_count == 0:
    print("\n  WARNING: No rows matched this landing_timestamp. "
          "Check that the timestamp value is correct.")
print("\n  Preview of the bronze data below:")

display(bronze_df)

## Step 3: Add audit columns
Two tracking columns are added to every row:

* `insert_timestamp`: when the row was **first** written to silver.
* `update_timestamp`: when the row was **last** written or changed in silver.

For `MERGE`, an existing row keeps its original `insert_timestamp` and only `update_timestamp` is refreshed (see Step 4).

In [0]:
from pyspark.sql import functions as F

updated_df = bronze_df.withColumn('insert_timestamp', F.current_timestamp()).withColumn('update_timestamp', F.current_timestamp())

print("=" * 70)
print("STEP 3: Added audit columns")
print("=" * 70)
print("  + insert_timestamp : when the row was first written to silver")
print("  + update_timestamp : when the row was last written/changed in silver")
print(f"  Total columns now  : {len(updated_df.columns)}")

## Step 4: Write to silver (FULL / APPEND / MERGE)
The target is `adb_db_caresync.silver.<table_name>`. What happens depends on `load_type` (upper/lower case and spaces don't matter):

* **FULL**: overwrite. The silver table ends up with **only** this batch.
* **APPEND**: add the rows to the end of the table. The table is created if it doesn't exist.
* **MERGE** (upsert on `merge_keys`):
  1. Checks that `merge_keys` is given and that every key column exists in the data. Otherwise it stops with a clear error.
  2. **First run** (table doesn't exist yet): just creates the table with this batch.
  3. **Later runs**:
     * Removes duplicate keys inside the batch, because MERGE fails if one target row matches two source rows.
     * Rows whose key **already exists**: all columns are updated **except** the key(s) and `insert_timestamp`.
     * Rows with a **new** key are inserted.
     * Prints how many rows were updated and inserted (read from the Delta table history).
* **Anything else**: stops with an "Invalid load_type" error.

In [0]:
from delta.tables import DeltaTable

target_table = f'adb_db_caresync.silver.{table_name}'
load_type_normalized = load_type.strip().upper()
table_exists = spark.catalog.tableExists(target_table)

print("=" * 70)
print("STEP 4: Write data to SILVER")
print("=" * 70)
print(f"  Target table      : {target_table}")
print(f"  Table exists?     : {'Yes' if table_exists else 'No (it will be created)'}")
print(f"  Load type         : {load_type_normalized or '(empty)'}")

if load_type_normalized == 'FULL':
    # Full load: overwrite the target table
    print("  Action            : FULL load -> replacing ALL existing rows in the silver table")
    updated_df.write.mode('overwrite').saveAsTable(target_table)
    print("  Result            : Silver table overwritten successfully")

elif load_type_normalized == 'APPEND':
    # Append: add new rows to the target table (creates it if missing)
    print("  Action            : APPEND -> adding new rows on top of existing data")
    updated_df.write.mode('append').saveAsTable(target_table)
    print("  Result            : Rows appended successfully")

elif load_type_normalized == 'MERGE':
    keys = [k.strip() for k in merge_keys.split(',') if k.strip()]
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = 'MERGE'")
    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")
    print(f"  Merge keys        : {keys}")

    if not table_exists:
        # First run: nothing to merge into, so create the table
        print("  Action            : MERGE (first run) -> table does not exist yet, creating it")
        updated_df.write.mode('overwrite').saveAsTable(target_table)
        print("  Result            : Silver table created successfully")
    else:
        # Deduplicate source on merge keys to avoid multiple-match merge errors
        source_df = updated_df.dropDuplicates(keys)
        print("  Action            : MERGE (upsert) -> update matching rows, insert new rows")
        print(f"  Source rows after removing duplicate keys: {source_df.count()}")
        merge_condition = ' AND '.join([f"t.`{k}` = s.`{k}`" for k in keys])

        # On update keep the original insert_timestamp; refresh all other columns
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c not in keys and c != 'insert_timestamp'}
        insert_values = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        (DeltaTable.forName(spark, target_table).alias('t')
            .merge(source_df.alias('s'), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_values)
            .execute())

        # Show what the merge did (from the Delta table history)
        metrics = (DeltaTable.forName(spark, target_table).history(1)
                   .select('operationMetrics').collect()[0][0]) or {}
        print("  Result            : MERGE completed successfully")
        print(f"    Rows updated    : {metrics.get('numTargetRowsUpdated', 'n/a')}")
        print(f"    Rows inserted   : {metrics.get('numTargetRowsInserted', 'n/a')}")

else:
    raise ValueError(f"Invalid load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

## Step 5: Quick check (optional)
A simple query to look at the result in silver. It is hard-coded to the `hospitals` table, so change the table name if you are loading something else, or skip this cell in production runs.

In [0]:
%sql
-- Optional sanity check: look at the data now sitting in a silver table
select * from adb_db_caresync.silver.hospitals

## Step 6: Return the record count
Counts how many rows were processed in this batch and sends that number back to the caller with `dbutils.notebook.exit(...)`.

A parent notebook or job (for example a loop that loads many tables) can read this value to log or check the load. **Keep this as the last cell**, because nothing after `exit` runs.

In [0]:
record_count = updated_df.count()

print("=" * 70)
print("STEP 6: Finished")
print("=" * 70)
print(f"  Records processed : {record_count}")
print(f"  From              : adb_db_caresync.bronze.{table_name}")
print(f"  To                : adb_db_caresync.silver.{table_name}")
print(f"  Load type         : {load_type.strip().upper()}")
print("  Returning the record count to the calling job/notebook.")
dbutils.notebook.exit(str(record_count))